# CS-781 Phase 6 (Abstractive Summarization) - data inspection

Run all on **CPU** (accelerator None), Internet on, secrets `HF` and `HF_WRITE`. Reads `Phase_6_Summ/Train/train.json` and `Phase_6_Summ/Test/test.json`, prints sizes, document and summary lengths (words and Phase-1 BPE tokens), how abstractive the summaries are, where in the judgment their content comes from, two examples, and BERTScore / ROUGE of simple copy baselines (lead / tail / head+tail of the document) on 150 training documents. About 10 minutes (BERTScore downloads roberta-large once). Send the whole printout back.

In [1]:
TOKENIZER_NAME = "a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json"
!pip install -q -U "huggingface_hub>=1.5"
import os, json, shutil, subprocess, sys, time, zipfile, random, math, gc
from collections import Counter, defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download, login, download_bucket_files, batch_bucket_files
from kaggle_secrets import UserSecretsClient

NB_START = time.time()
WORK = "/kaggle/working"
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_BUCKET = "kavincg23/cs781-models"
HF_CHECKPOINT_REPO = "kavincg23/cs781-checkpoints"   # old repo, read-only fallback
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
HF_WRITE_TOKEN = user_secrets.get_secret("HF_WRITE")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")


def get_dataset_file(filename):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN)


def fetch_model_file(name, local_path=None):
    local_path = local_path or f"{WORK}/{name}"
    if os.path.exists(local_path):
        return local_path
    part = local_path + ".part"
    try:
        download_bucket_files(HF_BUCKET, files=[(name, part)], raise_on_missing_files=True, token=HF_WRITE_TOKEN)
        os.replace(part, local_path)
        return local_path
    except Exception:
        if os.path.exists(part):
            os.remove(part)
    # fallback: the OLD model repo (the Phase-1 notebooks do the same; the tokenizer only lives there)
    p = hf_hub_download(repo_id=HF_CHECKPOINT_REPO, filename=name, repo_type="model", token=HF_WRITE_TOKEN)
    shutil.copy(p, local_path)
    print(f"(found {name} in the old repo {HF_CHECKPOINT_REPO})")
    return local_path


def upload_model_file(local_path, name=None):
    name = name or os.path.basename(local_path)
    try:
        batch_bucket_files(HF_BUCKET, add=[(local_path, name)], token=HF_WRITE_TOKEN)
        print(f"uploaded {name} to bucket {HF_BUCKET}")
    except Exception as e:
        print(f"(upload of {name} failed: {type(e).__name__}: {e}; the file is still in {WORK})")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 855.2/855.2 kB 3.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.3.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.3.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.3.0 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.3.0 which is incompatible.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.3.0 which is incompatible.
tokenizers 0.23.1 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.3.0 which is incompatible.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


device: cpu 


In [2]:
import array, heapq, time
from collections import Counter, defaultdict

BPE_MIN_WORD_FREQ = 1      # 1 = learn merges from every word type (exactly the Assignment-1 algorithm)
_END = "</w>"


class BPETokenizer:
    """Byte-level BPE (Assignment 1). Initial symbols are the UTF-8 bytes of each word, held as chr(b), plus
    '</w>' marking the end of a word so merges never cross word boundaries. Ties in pair frequency break
    lexicographically. Text is NOT lowercased."""

    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        self.merges = []
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.stoi = {}
        self._ranks = {}
        self._id_cache = {}

    # ---- helpers ----
    def _word_symbols(self, word):
        syms = [chr(b) for b in word.encode("utf-8")]
        syms.append(_END)
        return syms

    def _merge(self, symbols, pair):
        res, i = [], 0
        a, b = pair
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                res.append(a + b)
                i += 2
            else:
                res.append(symbols[i])
                i += 1
        return res

    # ---- training ----
    def fit_file(self, path, block_chars=50_000_000, min_word_freq=1):
        """Count word frequencies by streaming the corpus file in blocks, then learn the merges."""
        t0 = time.time()
        word_freqs = Counter()
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()          # the last word may be cut in half: finish it with the next block
                else:
                    leftover = ""
                word_freqs.update(parts)
        if leftover:
            word_freqs[leftover] += 1
        print(f"  counted {sum(word_freqs.values()):,} words ({len(word_freqs):,} distinct) in {time.time() - t0:.0f}s", flush=True)
        return self.fit_word_freqs(word_freqs, min_word_freq)

    def fit_word_freqs(self, word_freqs, min_word_freq=1):
        t0 = time.time()
        target = self.vocab_size
        words = [w for w, c in word_freqs.items() if c >= min_word_freq]
        freqs = [word_freqs[w] for w in words]
        splits = [self._word_symbols(w) for w in words]
        del word_freqs

        base = set()
        for syms in splits:
            base.update(syms)
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.merges = []
        self.itos.extend(sorted(base))
        print(f"  base byte alphabet: {len(base)} symbols; learning merges from {len(words):,} word types", flush=True)

        # pair_counts: how often each adjacent pair occurs (weighted by word frequency).
        # pair_words: which word types contain each pair, so a merge only revisits those words.
        pair_counts = {}
        pair_words = defaultdict(set)
        for i, syms in enumerate(splits):
            f = freqs[i]
            for j in range(len(syms) - 1):
                p = (syms[j], syms[j + 1])
                pair_counts[p] = pair_counts.get(p, 0) + f
                pair_words[p].add(i)

        # Max-heap of (-count, pair): pops the most frequent pair, ties -> lexicographically smallest pair
        # (identical to Assignment 1's min over (-count, pair)). Entries go stale when a count changes; a
        # fresh entry is pushed for every changed pair and stale ones are skipped when they reach the top.
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        while len(self.itos) < target:
            best, best_f = None, 0
            while heap:
                neg_c, p = heap[0]
                if pair_counts.get(p, 0) == -neg_c:
                    best, best_f = p, -neg_c
                    heapq.heappop(heap)
                    break
                heapq.heappop(heap)                  # stale entry
            if best is None:
                break
            if best_f < 2:
                print("  stopping: no pair occurs more than once", flush=True)
                break

            self.merges.append([best[0], best[1]])
            self.itos.append(best[0] + best[1])

            changed = set()
            for i in list(pair_words[best]):
                syms, f = splits[i], freqs[i]
                for j in range(len(syms) - 1):
                    p = (syms[j], syms[j + 1])
                    c = pair_counts.get(p, 0) - f
                    if c <= 0:
                        pair_counts.pop(p, None)
                    else:
                        pair_counts[p] = c
                    pair_words[p].discard(i)
                    changed.add(p)
                new_syms = self._merge(syms, best)
                splits[i] = new_syms
                for j in range(len(new_syms) - 1):
                    p = (new_syms[j], new_syms[j + 1])
                    pair_counts[p] = pair_counts.get(p, 0) + f
                    pair_words[p].add(i)
                    changed.add(p)
            for p in changed:
                c = pair_counts.get(p)
                if c:
                    heapq.heappush(heap, (-c, p))
            del pair_words[best]

            if len(self.merges) % 1000 == 0:
                print(f"  merges {len(self.merges):,} | vocab {len(self.itos):,} | {time.time() - t0:.0f}s", flush=True)

        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self._ranks = {(a, b): r for r, (a, b) in enumerate(self.merges)}
        self._id_cache = {}
        self.vocab_size = len(self.itos)      # the size the embedding table must actually have
        return self

    # ---- save / load (so the tokenizer is trained once and then reused) ----
    def save(self, path, meta=None):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"vocab_size": self.vocab_size, "merges": self.merges, "itos": self.itos,
                       "meta": meta or {}}, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(data["vocab_size"])
        tok.merges = [list(m) for m in data["merges"]]
        tok.itos = list(data["itos"])
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok._ranks = {(a, b): r for r, (a, b) in enumerate(tok.merges)}
        tok._id_cache = {}
        tok.vocab_size = len(tok.itos)
        return tok

    # ---- inference ----
    def _tokenize_word(self, word):
        syms = self._word_symbols(word)
        while len(syms) > 1:
            best_rank, best_j = None, -1
            for j in range(len(syms) - 1):
                r = self._ranks.get((syms[j], syms[j + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_j = r, j
            if best_rank is None:
                break
            syms = self._merge(syms, (syms[best_j], syms[best_j + 1]))
        return syms

    def _word_ids(self, word):
        ids = self._id_cache.get(word)
        if ids is None:
            unk = self.stoi["<unk>"]
            ids = [self.stoi.get(t, unk) for t in self._tokenize_word(word)]     # symbols never seen in training -> <unk>
            self._id_cache[word] = ids
        return ids

    def tokenize(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._tokenize_word(word))
        return out

    def get_vocab_size(self):
        return len(self.itos)

    def encode(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._word_ids(word))
        return out

    def encode_file(self, path, block_chars=5_000_000):
        """Encode a whole text file into a packed array of ids, block by block (bounded memory)."""
        out = array.array("i")
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()
                else:
                    leftover = ""
                cache = self._id_cache
                for word in parts:
                    ids = cache.get(word)
                    if ids is None:
                        ids = self._word_ids(word)
                    out.extend(ids)
        if leftover:
            out.extend(self._word_ids(leftover))
        return out

    def decode(self, ids):
        """ids -> string. Words are re-joined with single spaces; <unk> decodes to '?'."""
        words, cur = [], bytearray()
        for i in ids:
            i = int(i)
            if not 0 <= i < len(self.itos):
                continue
            tok = self.itos[i]
            if tok in ("<pad>", "<s>", "</s>"):
                continue
            if tok == "<unk>":
                cur.extend(b"?")
                continue
            ends_word = tok.endswith(_END)
            if ends_word:
                tok = tok[: -len(_END)]
            cur.extend(ord(c) & 0xFF for c in tok)
            if ends_word:
                words.append(bytes(cur).decode("utf-8", errors="replace"))
                cur = bytearray()
        if cur:
            words.append(bytes(cur).decode("utf-8", errors="replace"))
        return " ".join(words)



bpe_tokenizer = BPETokenizer.load(fetch_model_file(TOKENIZER_NAME))
VOCAB = bpe_tokenizer.get_vocab_size()
print("tokenizer:", TOKENIZER_NAME, "| vocab", VOCAB)

(…)full_v40000_160000000w_seed42_minf1.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

(found a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json in the old repo kavincg23/cs781-checkpoints)
tokenizer: a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json | vocab 40000


In [3]:
import numpy as np, re, random, collections, subprocess, sys
P6 = "Phase_6_Summ"
def load_json(p):
    with open(p, encoding="utf-8") as f:
        return json.load(f)
TR = load_json(get_dataset_file(f"{P6}/Train/train.json")); TE = load_json(get_dataset_file(f"{P6}/Test/test.json"))
def as_list(x): return list(x.values()) if isinstance(x, dict) else x
TR, TE = as_list(TR), as_list(TE)
print(f"train: {len(TR)} records, fields {list(TR[0].keys())} | test: {len(TE)} records, fields {list(TE[0].keys())}")
DK = next(k for k in TR[0] if k.lower() in ("document", "text", "doc", "judgment", "judgement"))
SK = next(k for k in TR[0] if k.lower() in ("summary", "abstract", "headnote"))
IK = next(k for k in TR[0] if k.lower() in ("id", "doc_id", "case_id"))
print("id / document / summary fields:", IK, DK, SK)
def txt(x): return " ".join(x) if isinstance(x, list) else str(x)
print("ids unique: train", len({r[IK] for r in TR}) == len(TR), "| test", len({r[IK] for r in TE}) == len(TE), "| train/test id overlap", len({r[IK] for r in TR} & {r[IK] for r in TE}))
H = lambda t: hash(t[:2000] + t[-2000:])
print("same document text train/test:", len({H(txt(r[DK])) for r in TR} & {H(txt(r[DK])) for r in TE}), "| duplicated documents inside train:", len(TR) - len({H(txt(r[DK])) for r in TR}),
      "| empty summaries:", sum(1 for r in TR if not txt(r[SK]).strip()), "| empty documents: train", sum(1 for r in TR if not txt(r[DK]).strip()), "test", sum(1 for r in TE if not txt(r[DK]).strip()))
def pct(a, ps=(1, 5, 25, 50, 75, 90, 95, 99, 100)):
    a = sorted(a); return {p: a[min(len(a) - 1, int(p / 100 * len(a)))] for p in ps}
dw = [len(txt(r[DK]).split()) for r in TR]; sw = [len(txt(r[SK]).split()) for r in TR]; tw = [len(txt(r[DK]).split()) for r in TE]
print("\nWORDS  train documents:", pct(dw)); print("       train summaries:", pct(sw)); print("       test documents: ", pct(tw))
ratio = np.array(sw) / np.maximum(np.array(dw), 1)
print("summary / document length ratio:", {k: round(float(v), 3) for k, v in pct(ratio.tolist()).items()}, "| correlation of lengths:", round(float(np.corrcoef(dw, sw)[0, 1]), 3))
rng = random.Random(0); S = rng.sample(range(len(TR)), min(1000, len(TR)))
dt = [len(bpe_tokenizer.encode(txt(TR[i][DK]))) for i in S]; st = [len(bpe_tokenizer.encode(txt(TR[i][SK]))) for i in S]
print(f"\nBPE TOKENS (sample of {len(S)}) documents:", pct(dt)); print("                         summaries:", pct(st))
print(f"tokens per word: documents {np.sum(dt) / np.sum([dw[i] for i in S]):.2f}, summaries {np.sum(st) / np.sum([sw[i] for i in S]):.2f}")
for cap in (256, 384, 512, 768, 1024, 1536):
    print(f"   summaries <= {cap:5d} tokens: {np.mean([t <= cap for t in st]):.3f}")
for cap in (512, 1024, 1536, 2048, 4096, 8192, 16384):
    print(f"   documents <= {cap:5d} tokens: {np.mean([t <= cap for t in dt]):.3f}")
# how abstractive: share of summary n-grams not in the document; where in the document the summary's content comes from
def toks(t): return re.findall(r"\w+", t.lower())
nov = {1: [], 2: [], 3: []}; posh = np.zeros(10); sent_first = []
for i in S[:500]:
    d, s = toks(txt(TR[i][DK])), toks(txt(TR[i][SK]))
    if not d or not s: continue
    for n in nov:
        dn = set(zip(*[d[k:] for k in range(n)])); sn = list(zip(*[s[k:] for k in range(n)]))
        if sn: nov[n].append(np.mean([g not in dn for g in sn]))
    first = {}
    for k in range(len(d) - 2):
        first.setdefault(tuple(d[k:k + 3]), k)
    hits = [first[g] / len(d) for g in zip(s, s[1:], s[2:]) if g in first]
    for h in hits: posh[min(int(h * 10), 9)] += 1
print("\nnovel n-grams in the summary (not in the document; 0 = purely extractive): " + ", ".join(f"{n}-gram {np.mean(v):.3f}" for n, v in nov.items()))
print("where the summary's 3-grams first occur in the document (share per decile, 0-10% ... 90-100%):", [round(float(x), 3) for x in posh / max(posh.sum(), 1)])
# examples
for i in S[:2]:
    r = TR[i]; print(f"\n--- id {r[IK]}: document {len(txt(r[DK]).split())} words, summary {len(txt(r[SK]).split())} words\nDOC START: {txt(r[DK])[:600]!r}\nDOC END:   {txt(r[DK])[-400:]!r}\nSUMMARY:   {txt(r[SK])[:900]!r}")
# baselines on 150 training documents: lead-N words (N = the median summary length) and the summary-length-matched random-sentence baseline
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert-score", "rouge-score"], check=False)
from rouge_score import rouge_scorer
from bert_score import score as bscore
B = S[:150]; med = int(np.median(sw)); rs = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
refs = [txt(TR[i][SK]) for i in B]
def lead(i, n): return " ".join(txt(TR[i][DK]).split()[:n])
def tail(i, n): return " ".join(txt(TR[i][DK]).split()[-n:])
def ht(i, n): w = txt(TR[i][DK]).split(); return " ".join(w[:n // 2] + w[-(n - n // 2):]) if len(w) > n else " ".join(w)
BASES = {f"lead-{med}": [lead(i, med) for i in B], f"tail-{med}": [tail(i, med) for i in B], f"head+tail-{med}": [ht(i, med) for i in B],
         f"lead-{med // 2}": [lead(i, med // 2) for i in B], f"lead-{2 * med}": [lead(i, 2 * med) for i in B]}
print(f"\nBASELINES on {len(B)} training documents (median summary length {med} words). BERTScore = roberta-large, F1 (bert-score default, no rescaling):")
BL = {}
for name, hyp in BASES.items():
    P_, R_, F_ = bscore(hyp, refs, lang="en", model_type="roberta-large", verbose=False, batch_size=16)
    r = [rs.score(a, b) for a, b in zip(refs, hyp)]
    BL[name] = dict(bertscore_f1=float(F_.mean()), rouge1=float(np.mean([x["rouge1"].fmeasure for x in r])), rougeL=float(np.mean([x["rougeL"].fmeasure for x in r])))
    print(f"  {name:18s} BERTScore F1 {BL[name]['bertscore_f1']:.4f}  ROUGE-1 {BL[name]['rouge1']:.4f}  ROUGE-L {BL[name]['rougeL']:.4f}")
json.dump(dict(n_train=len(TR), n_test=len(TE), fields=[IK, DK, SK], doc_words=pct(dw), sum_words=pct(sw), test_words=pct(tw), doc_tokens=pct(dt), sum_tokens=pct(st), baselines=BL),
          open(f"{WORK}/p6_data_inspection.json", "w"), indent=1, default=str)
print("\nsend me this whole printout")

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

train: 7030 records, fields ['id', 'document', 'summary'] | test: 100 records, fields ['id', 'document']
id / document / summary fields: id document summary
ids unique: train True | test True | train/test id overlap 0
same document text train/test: 0 | duplicated documents inside train: 28 | empty summaries: 2 | empty documents: train 0 test 0

WORDS  train documents: {1: 570, 5: 1041, 25: 2017, 50: 3129, 75: 4972, 90: 7855, 95: 11328, 99: 24268, 100: 139645}
       train summaries: {1: 127, 5: 232, 25: 421, 50: 638, 75: 963, 90: 1501, 95: 2011, 99: 4254, 100: 27768}
       test documents:  {1: 573, 5: 1325, 25: 2100, 50: 3201, 75: 5519, 90: 9344, 95: 14132, 99: 39695, 100: 39695}
summary / document length ratio: {1: 0.039, 5: 0.071, 25: 0.141, 50: 0.217, 75: 0.311, 90: 0.396, 95: 0.443, 99: 0.555, 100: 9.151} | correlation of lengths: 0.748

BPE TOKENS (sample of 1000) documents: {1: 591, 5: 1090, 25: 2206, 50: 3350, 75: 5360, 90: 8313, 95: 11792, 99: 32584, 100: 74970}
              

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

... (16 outputs left out)

  lead-319           BERTScore F1 0.8299  ROUGE-1 0.3806  ROUGE-L 0.1944


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  lead-1276          BERTScore F1 0.8330  ROUGE-1 0.4998  ROUGE-L 0.2797

send me this whole printout
